In [1]:
import scanpy as sc
import treedata as td
import pycea as py
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import pandas as pd

# Import utility functions made by Katie
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, '/project/imoskowitz/yubin/SmoNull_NMPs_mesoderm_biased_analysis')

from src.I_preprocessing.plot_preprocessing import plot_UMAP_custom

from lineage_utilities.tree_functions import *

In [2]:
import matplotlib.colors as mcolors
palette_template = mcolors.TABLEAU_COLORS

In [3]:
data_dir = "output_data"
plot_dir = "output_plot"
base_path = "/project/imoskowitz/yubin/Lineage_Tree_Construction/"
output_path_data = base_path+data_dir+"/"
output_path_plot = base_path+plot_dir+"/Trees/Robin_Pijuan/Manual_Cardiac_Annotation/E8_5/"
adata_fname = "Processed_data/E8_5.h5td"

In [4]:
adata = td.read_h5td(output_path_data+"Processed_data/E8_5.h5td")


In [5]:
adata.obs['cardiac_labels'].unique()

[NaN, 'pSHF', 'aSHF', 'Differentiating CM', 'CMs', 'JCF']
Categories (5, object): ['CMs', 'Differentiating CM', 'JCF', 'aSHF', 'pSHF']

In [6]:
adata.X = adata.layers["raw_counts"]

In [7]:
adata.obs['cardiac_labels'] = adata.obs['cardiac_labels'].astype('category')
adata.obs['tree'] = adata.obs['tree'].astype('category')
adata.obs['germ_layer'] = adata.obs['germ_layer'].astype('category')

In [8]:
tree_sizes = adata.obs['tree'].value_counts().sort_values(ascending=False)
print(tree_sizes.head(10))

# flag trees likely to be memory-heavy (dense O(n^2) LCA matrix)
LARGE_TREE_THRESHOLD = 2000  # adjust based on what crashed before
large_trees = tree_sizes[tree_sizes > LARGE_TREE_THRESHOLD].index.tolist()
print("Large trees (may need special handling):", large_trees)

tree
E8.5-R3-C1    19886
E8.5-R2-C1    15318
E8.5-R2-C2    12352
E8.5-R1-C1     6192
E8.5-R3-C2     4202
E8.5-R1-C3     2955
E8.5-R1-C2     2256
E8.5-R3-C3      664
E8.5-R1-C4      429
Name: count, dtype: int64
Large trees (may need special handling): ['E8.5-R3-C1', 'E8.5-R2-C1', 'E8.5-R2-C2', 'E8.5-R1-C1', 'E8.5-R3-C2', 'E8.5-R1-C3', 'E8.5-R1-C2']


#### Pick a subset of cells that you want to plot

In [ ]:
# E8.5 with Colgan Auto Annotation
# Cardiac_cell_types = {"Juxtacardiac field" : "Juxtacardiac field",
#                         "First heart field": "First heart field",
#                         "Second heart field": "Second heart field",
#                         "Anterior mesoderm": "anterior second heart field" 
#                       }

### Functions

#### Debug memory problem

In [ ]:
clone_tdata = adata[adata.obs["tree"] == "E8.5-R3-C1"].copy()
print("subset done")

points, depth_count, leaf_order, valid_types = compute_lca_depth_points(
    clone_tdata, "E8.5-R3-C1", color="cardiac_labels", n_leaf=5
)
print("LCA computation done")  # if it dies before this prints, it's the matrix

# only then try plotting
result = plot_tree_custom(clone_tdata, "E8.5-R3-C1", color="cardiac_labels", output_path_plot=output_path_plot)
print("plotting done")

In [ ]:
import gc

output_csv = output_path_plot + "depth_points_all_trees.csv"
csv_path = Path(output_csv)
if csv_path.exists():
    csv_path.unlink()  # start fresh; remove if you want to append across runs

for clone_key in adata.obs['tree'].unique():
    if pd.isna(clone_key) or clone_key == 'nan': # To catch cells that do not have a tree
        continue

    n_cells = tree_sizes.get(clone_key, 0)
    print(f"starting {clone_key} ({n_cells} cells)")

    clone_tdata = adata[adata.obs["tree"] == clone_key].copy()

    # plotting + LCA computation happen once, together
    result = plot_tree_custom(clone_tdata, clone_key, color="cardiac_labels", output_path_plot=output_path_plot)

    points = result["points"]
    leaf_order = result["leaf_order"]

    rows = [{"tree": clone_key, "cell_type": ct, "depth": d, "pos": p, "n_leaves": len(leaf_order)}
            for d, p, ct in points]
    df_chunk = pd.DataFrame(rows)
    df_chunk.to_csv(output_csv, mode='a', header=not csv_path.exists(), index=False)

    # explicit cleanup
    del clone_tdata, result, points, df_chunk
    gc.collect()

    print(f"complete {clone_key}")

plt.close('all')  # belt-and-suspenders in case anything else left a figure open

#### Redo this plot to show more information regarding cell count per clone, proportion of cells that are cardiac, and cell count of every cardiac type

In [ ]:
ct = pd.crosstab(adata.obs['tree'], adata.obs['cardiac_labels'], normalize='index') * 100
ct.round(1)

In [ ]:
palette = get_color_palette(adata, color_key='cardiac_labels')
colors = [palette.get(ct, "gray") for ct in ct.columns]

ax = ct.plot(kind='bar', stacked=True, figsize=(14, 6), color=colors, width=0.85)
ax.set_ylabel("% of cells", size = 14)
ax.set_xlabel("Tree", size = 14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha='center', fontsize=14)
ax.set_title("Cell type composition per tree", size = 24)
ax.legend(title="Cell type", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(output_path_plot + "cardiac_celltype_composition_per_tree.svg", bbox_inches='tight')